In [15]:
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

In [16]:
train= pd.read_csv("/kaggle/input/competitions/for-house-prices-advanced-regression-techniques-you-can-us/train.csv")

test=pd.read_csv("/kaggle/input/competitions/for-house-prices-advanced-regression-techniques-you-can-us/test.csv")

sub=pd.read_csv("/kaggle/input/competitions/for-house-prices-advanced-regression-techniques-you-can-us/sample_submission.csv")

In [17]:
train.head()

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
0,0,1,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1
1,1,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1
2,2,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1
3,3,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1
4,4,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0


In [18]:
train['windspeed'] = train['windspeed'].fillna(train['windspeed'].mode()[0])

test['winddirection'] = test['winddirection'].fillna(test['winddirection'].mode()[0])

In [19]:
train.isnull().sum()

id               0
day              0
pressure         0
maxtemp          0
temparature      0
mintemp          0
dewpoint         0
humidity         0
cloud            0
sunshine         0
winddirection    0
windspeed        0
rainfall         0
dtype: int64

In [20]:
test.head()

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed
0,2190,1,1019.5,17.5,15.8,12.7,14.9,96.0,99.0,0.0,50.0,24.3
1,2191,2,1016.5,17.5,16.5,15.8,15.1,97.0,99.0,0.0,50.0,35.3
2,2192,3,1023.9,11.2,10.4,9.4,8.9,86.0,96.0,0.0,40.0,16.9
3,2193,4,1022.9,20.6,17.3,15.2,9.5,75.0,45.0,7.1,20.0,50.6
4,2194,5,1022.2,16.1,13.8,6.4,4.3,68.0,49.0,9.2,20.0,19.4


In [21]:
test.shape

(730, 12)

In [22]:
test.isnull().sum()

id               0
day              0
pressure         0
maxtemp          0
temparature      0
mintemp          0
dewpoint         0
humidity         0
cloud            0
sunshine         0
winddirection    0
windspeed        0
dtype: int64

In [23]:
sub.head()

,id,rainfall
0,2190,0
1,2191,0
2,2192,0
3,2193,0
4,2194,0


In [24]:
sub.shape

(730, 2)

In [25]:
def create_features(df):
    df = df.copy()
    
    df['day_sin'] = np.sin(2 * np.pi * df['day'] / 365.25)
    df['day_cos'] = np.cos(2 * np.pi * df['day'] / 365.25)
    
    df['temp_range'] = df['maxtemp'] - df['mintemp']
    df['dew_spread'] = df['temparature'] - df['dewpoint']
    
    wind_rad = np.deg2rad(df['winddirection'])
    df['wind_u'] = df['windspeed'] * np.cos(wind_rad)
    df['wind_v'] = df['windspeed'] * np.sin(wind_rad)
    
    df['humidity_temp_inter'] = df['humidity'] * df['temparature']
    df['cloud_sun_diff'] = df['cloud'] - df['sunshine']
    
    return df

train_fe = create_features(train)
test_fe = create_features(test)

In [26]:
X = train_fe.drop(columns=['id', 'rainfall'])
y = train_fe['rainfall']
X_test = test_fe.drop(columns=['id'])

In [27]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_valid_scaled = scaler.transform(X_valid)
    X_test_scaled = scaler.transform(X_test)
    
    rf = RandomForestClassifier(n_estimators=1000, random_state=42, n_jobs=-1)
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lgbm = LGBMClassifier(n_estimators=1000, learning_rate=0.05, random_state=42, verbose=-1)
    
    rf.fit(X_train, y_train)
    lr.fit(X_train_scaled, y_train)
    lgbm.fit(X_train, y_train)
    
    p_rf = rf.predict_proba(X_valid)[:, 1]
    p_lr = lr.predict_proba(X_valid_scaled)[:, 1]
    p_lgbm = lgbm.predict_proba(X_valid)[:, 1]
    
    oof_preds[valid_idx] = (p_rf + p_lr + p_lgbm) / 3
    
    test_preds += (
        rf.predict_proba(X_test)[:, 1] + 
        lr.predict_proba(X_test_scaled)[:, 1] + 
        lgbm.predict_proba(X_test)[:, 1]
    ) / 30
    
    fold_auc = roc_auc_score(y_valid, oof_preds[valid_idx])
    print(f"Fold {fold + 1} finished | Validation AUC: {fold_auc:.5f}")

print(f"\nOverall Out-Of-Fold ROC AUC: {roc_auc_score(y, oof_preds):.5f}")

Fold 1 finished | Validation AUC: 0.88227
Fold 2 finished | Validation AUC: 0.95174
Fold 3 finished | Validation AUC: 0.88103
Fold 4 finished | Validation AUC: 0.81324
Fold 5 finished | Validation AUC: 0.87520
Fold 6 finished | Validation AUC: 0.87598
Fold 7 finished | Validation AUC: 0.89439
Fold 8 finished | Validation AUC: 0.91863
Fold 9 finished | Validation AUC: 0.91515
Fold 10 finished | Validation AUC: 0.87565

Overall Out-Of-Fold ROC AUC: 0.88776


In [28]:
submission = pd.DataFrame({'id': test['id'], 'rainfall': test_preds})
submission.to_csv('submission.csv', index=False)
display(submission.head())

,id,rainfall
0,2190,0.989806
1,2191,0.993532
2,2192,0.930230
3,2193,0.085260
4,2194,0.064320
